# Anomaly detection 3/3 — Autoencoder reconstruction error

TinyML course, Module 8. Adapted from the original course notebook (`Autoencoder.ipynb`).

K-means and GMM scored *feature vectors*. An **autoencoder** works on the **raw window**: a network is trained to reproduce its own input through a narrow bottleneck. It only learns to do that for data resembling the training data — so the **reconstruction error** `MSE(x, x̂)` is an anomaly score that needs no feature engineering.

Plan:
1. Synthetic "vibration" windows (as in the original notebook) — the clean picture
2. Your fan windows: train on `normal` only, score the fault states
3. Reality check: does this fit on an nRF52840?

## Part 1 — Synthetic vibration data

Normal = ~50 Hz sine with jitter and mild noise. Anomalous = shifted to ~80 Hz, noisier, with impulsive spikes (think: scraping blade).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
fs, T = 1000, 0.2
n_samples = int(fs * T)               # 200 samples per window
t = np.linspace(0, T, n_samples, endpoint=False)

def normal_window():
    f = 50 + np.random.uniform(-2, 2)
    A = 1.0 + np.random.uniform(-0.2, 0.2)
    return A * np.sin(2 * np.pi * f * t) + 0.05 * np.random.randn(n_samples)

def anomalous_window():
    f = 80 + np.random.uniform(-5, 5)
    A = 1.0 + np.random.uniform(-0.2, 0.2)
    x = A * np.sin(2 * np.pi * f * t) + 0.2 * np.random.randn(n_samples)
    for _ in range(3):
        i = np.random.randint(0, n_samples)
        x[i:i + 2] += np.random.uniform(3, 5)
    return x

X_train = np.stack([normal_window() for _ in range(2000)])[..., None]
X_test = np.concatenate([np.stack([normal_window() for _ in range(400)]),
                         np.stack([anomalous_window() for _ in range(100)])])[..., None]
y_test = np.r_[np.zeros(400), np.ones(100)]

fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].plot(t, X_train[0, :, 0]); ax[0].set_title('normal window')
ax[1].plot(t, X_test[420, :, 0]); ax[1].set_title('anomalous window')
plt.tight_layout(); plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_conv_ae(timesteps, channels=1):
    inputs = keras.Input(shape=(timesteps, channels))
    x = layers.Conv1D(16, 7, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling1D(2, padding='same')(x)
    x = layers.Conv1D(32, 7, padding='same', activation='relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    x = layers.Conv1D(64, 7, padding='same', activation='relu')(x)
    encoded = layers.MaxPooling1D(2, padding='same', name='bottleneck')(x)
    x = layers.Conv1D(64, 7, padding='same', activation='relu')(encoded)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(32, 7, padding='same', activation='relu')(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(16, 7, padding='same', activation='relu')(x)
    x = layers.UpSampling1D(2)(x)
    decoded = layers.Conv1D(channels, 7, padding='same')(x)
    return keras.Model(inputs, decoded)

ae = build_conv_ae(n_samples)
ae.compile(optimizer='adam', loss='mse')
ae.summary()

# Train ONLY on normal windows: input == target
hist = ae.fit(X_train, X_train, epochs=30, batch_size=64,
              validation_split=0.1, verbose=1)

In [ ]:
X_pred = ae.predict(X_test)
errs = np.mean((X_pred - X_test) ** 2, axis=(1, 2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(errs); ax[0].axvline(400, ls='--', c='k')
ax[0].set_title('Reconstruction error (normal | anomalous)')
ax[0].set_xlabel('test window'); ax[0].set_ylabel('MSE')
ax[1].hist(errs[y_test == 0], bins=40, alpha=0.7, label='normal')
ax[1].hist(errs[y_test == 1], bins=40, alpha=0.7, label='anomalous')
ax[1].legend(); ax[1].set_title('Error distributions')
plt.tight_layout(); plt.show()

# Look at what the AE actually does to an anomaly: it redraws it as the
# nearest normal-looking signal - the residual is the fault signature.
idx = 405
plt.figure(figsize=(8, 3))
plt.plot(t, X_test[idx, :, 0], label='original (anomalous)')
plt.plot(t, X_pred[idx, :, 0], '--', label='reconstruction')
plt.legend(); plt.title(f'MSE = {errs[idx]:.4f}'); plt.tight_layout(); plt.show()

## Part 2 — Your fan windows

We reuse the raw 500-sample windows (2 s at 250 Hz) from the Module 7 loader. To keep the model 1-channel we feed the **resultant magnitude** `sqrt(x²+y²+z²)` (mean-removed) — orientation-independent, and spikes/harmonics survive.

Protocol as before: train on 70 % of `normal`, score everything else, held-out fault stays unseen until scoring.

In [ ]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath('../../module7-models/rf-features'))
from train_rf import load_windows, WINDOW   # noqa: E402

DATA_DIR = '../../module7-models/rf-features/data/raw'
HELD_OUT = 'scrape'   # keep identical to notebooks 1-2

X_raw, X_feat, y, _ = load_windows(DATA_DIR)

# resultant magnitude, per-window mean removed, global std normalised
mag = np.sqrt(np.sum(X_raw ** 2, axis=2))
mag = mag - mag.mean(axis=1, keepdims=True)

idx_normal = np.where(y == 'normal')[0]
rng = np.random.default_rng(0)
rng.shuffle(idx_normal)
train_idx = idx_normal[:int(0.7 * len(idx_normal))]
scale = mag[train_idx].std()
mag = (mag / scale)[..., None]
print('windows:', mag.shape, ' train (normal only):', len(train_idx))

In [ ]:
ae_fan = build_conv_ae(WINDOW)
ae_fan.compile(optimizer='adam', loss='mse')
ae_fan.fit(mag[train_idx], mag[train_idx], epochs=40, batch_size=32,
           validation_split=0.1, verbose=1)

recon = ae_fan.predict(mag)
errs = np.mean((recon - mag) ** 2, axis=(1, 2))
thr = np.percentile(errs[train_idx], 98)
print(f'threshold (98th pct of normal training error): {thr:.5f}')

In [ ]:
test_mask = np.ones(len(y), dtype=bool)
test_mask[train_idx] = False
df = pd.DataFrame({'state': y[test_mask], 'err': errs[test_mask]})

ax = df.boxplot(column='err', by='state', figsize=(8, 4))
ax.axhline(thr, color='r', ls='--')
ax.set_yscale('log')
plt.suptitle(''); plt.title('AE reconstruction error per fan state'); plt.show()

print('Fraction flagged per state (held-out =', HELD_OUT, '):')
for state, grp in df.groupby('state'):
    print(f"  {state:12s} {np.mean(grp['err'] > thr):5.1%}")

## Part 3 — Would this fit on the nRF52840?

Count the cost before falling in love:

- this Conv1D AE ≈ **65 k parameters ≈ 254 KB float32** — weights alone would not leave room in 256 KB RAM, and activations come on top
- int8 (Module 9!) → ~64 KB weights: possible but still the most expensive detector of the three
- vs the K-means block you deploy in Track 2: a few hundred bytes and microseconds

**Exercises**
1. Shrink it: build a *dense* autoencoder on the **13 features** instead (13 → 8 → 3 → 8 → 13, ~500 params). Compare its detection table with the Conv1D AE and with K-means/GMM. Is the raw-window advantage real *on your data*?
2. Plot reconstructions of a held-out-fault window. What part of the waveform does the AE fail to redraw — and does that match the physics of the fault?
3. Rank all three methods for the fan use-case on: detection rate, false alarms, MCU cost, explainability. Which one goes in the product, and which one stays in the lab as a labelling aid?

**Wrap-up:** all three detectors share one design: *learn normal, threshold the deviation score*. What changes is the score — distance, likelihood, reconstruction error — and the price you pay on-device. Now go do Track 2 and put the cheap one on the fan.